# EXERCISE 1

### Import the required modules

In [80]:
import tensorflow as tf
import os
from reader import AudioReader
from preprocessing import Padding, Normalization
from preprocessing import MelSpectrogram

In [81]:
# Check if GPU is detected
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Define the Hyperparameters

In [82]:
SCRIPT_DIR = os.path.abspath('')

PREPROCESSING_ARGS = {
    'sampling_rate': 16000,
    'frame_length_in_s': 0.032,
    'frame_step_in_s': 0.016,
    'num_mel_bins': 40,
    'lower_frequency': 20,
    'upper_frequency': 8000,
} # 83% [16000, 0.008, 0.002, 40, 20, 4000] # 97% [16000, 0.032, 0.016, 20, 20, 4000] # 91.5% [16000, 0.032, 0.016, 10, 20, 8000] # 98% [16000, 0.032, 0.016, 20, 20, 8000]

TRAINING_ARGS = {
    'batch_size': 20,
    'learning_rate': 0.01,
    'end_learning_rate': 1.e-5,
    'epochs': 20
}

In [83]:
os.path.join(SCRIPT_DIR, 'msc-train/down*')

'/home/gab/Documents/ML4IOT-HW/Lab_03/msc-train/down*'

### Create train/val/test Datasets

In [84]:
train_ds = tf.data.Dataset.list_files([os.path.join(SCRIPT_DIR, 'msc-train/down*'), os.path.join(SCRIPT_DIR, 'msc-train/up*')])
val_ds = tf.data.Dataset.list_files([os.path.join(SCRIPT_DIR, 'msc-val/down*'), os.path.join(SCRIPT_DIR, 'msc-val/up*')])
test_ds = tf.data.Dataset.list_files([os.path.join(SCRIPT_DIR, 'msc-test/down*'), os.path.join(SCRIPT_DIR, 'msc-test/up*')])

### Define the Data Pipeline

In [85]:
audio_reader = AudioReader(tf.int16)
padding = Padding(PREPROCESSING_ARGS['sampling_rate'])
normalization = Normalization(tf.int16)
mel_spec_processor = MelSpectrogram(**PREPROCESSING_ARGS)

LABELS = ['down', 'up']

def prepare_for_training(feature, label):
    feature = tf.expand_dims(feature, -1)
    label_id = tf.argmax(label == LABELS)

    return feature, label_id

train_ds = (train_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(mel_spec_processor.get_mel_spec_and_label)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size'])
            .cache())
val_ds = (val_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(mel_spec_processor.get_mel_spec_and_label)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size']))
test_ds = (test_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(mel_spec_processor.get_mel_spec_and_label)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size']))

### Read a batch of data

In [86]:
for example_batch, example_labels in train_ds.take(1):
  print('Batch Shape:', example_batch.shape)
  print('Data Shape:', example_batch.shape[1:])
  print('Labels:', example_labels)

Batch Shape: (20, 61, 40, 1)
Data Shape: (61, 40, 1)
Labels: tf.Tensor([1 0 1 0 0 1 0 1 1 0 1 0 1 1 0 0 0 0 1 0], shape=(20,), dtype=int64)


2024-11-28 01:28:30.740971: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


### Create the Model

In [87]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=example_batch.shape[1:]),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[2, 2], use_bias=False, padding='valid'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[1, 1], use_bias=False, padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Conv2D(filters=128, kernel_size=[3, 3], strides=[1, 1], use_bias=False, padding='same'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(units=len(LABELS)),
    tf.keras.layers.Softmax()
])

In [88]:
model.summary()

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_40 (Conv2D)          (None, 30, 19, 128)       1152      
                                                                 
 batch_normalization_33 (Ba  (None, 30, 19, 128)       512       
 tchNormalization)                                               
                                                                 
 re_lu_33 (ReLU)             (None, 30, 19, 128)       0         
                                                                 
 conv2d_41 (Conv2D)          (None, 30, 19, 128)       147456    
                                                                 
 batch_normalization_34 (Ba  (None, 30, 19, 128)       512       
 tchNormalization)                                               
                                                                 
 re_lu_34 (ReLU)             (None, 30, 19, 128)      

### Create callbacks

In [89]:
# Learning Rate scheduler
linear_decay = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=TRAINING_ARGS['learning_rate'],
    end_learning_rate=TRAINING_ARGS['end_learning_rate'],
    decay_steps=len(train_ds) * TRAINING_ARGS['epochs'],
)
lr_scheduler = tf.keras.callbacks.LearningRateScheduler(linear_decay)

# Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    verbose=1,
    mode='auto'
)

### Train the Model (with Callbacks)

- Modify the `learning_rate` argument of the `Adam` optimizer.
- Modify the `fit` method, specifying the `callbacks` argument.

In [90]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=False)
optimizer = tf.optimizers.Adam(learning_rate=linear_decay)
metrics = [tf.metrics.SparseCategoricalAccuracy()]
model.compile(loss=loss, optimizer=optimizer, metrics=metrics)

history = model.fit(
    train_ds, 
    epochs=TRAINING_ARGS['epochs'], 
    validation_data=val_ds, 
    callbacks=[lr_scheduler, early_stopping]
)

Epoch 1/20
80/80 [==============================] - 4s 21ms/step - loss: 0.6488 - sparse_categorical_accuracy: 0.6919 - val_loss: 0.6299 - val_sparse_categorical_accuracy: 0.7700 - lr: 0.0095
Epoch 2/20
80/80 [==============================] - 1s 18ms/step - loss: 0.4958 - sparse_categorical_accuracy: 0.8012 - val_loss: 0.3992 - val_sparse_categorical_accuracy: 0.8600 - lr: 0.0090
Epoch 3/20
80/80 [==============================] - 1s 17ms/step - loss: 0.3934 - sparse_categorical_accuracy: 0.8462 - val_loss: 0.3609 - val_sparse_categorical_accuracy: 0.8450 - lr: 0.0085
Epoch 4/20
80/80 [==============================] - 1s 17ms/step - loss: 0.3231 - sparse_categorical_accuracy: 0.8888 - val_loss: 1.1332 - val_sparse_categorical_accuracy: 0.7100 - lr: 0.0080
Epoch 5/20
80/80 [==============================] - 1s 18ms/step - loss: 0.2829 - sparse_categorical_accuracy: 0.9094 - val_loss: 3.5409 - val_sparse_categorical_accuracy: 0.5550 - lr: 0.0075
Epoch 6/20
80/80 [======================

### Show the History

In [91]:
history.history

{'loss': [0.648820698261261,
  0.4957592487335205,
  0.3933752477169037,
  0.32313793897628784,
  0.282937616109848,
  0.25534725189208984,
  0.2288123518228531,
  0.1952439546585083,
  0.17944392561912537,
  0.1482829451560974,
  0.13670994341373444,
  0.1176077201962471,
  0.10130447149276733,
  0.09124229103326797,
  0.0819811224937439,
  0.07479099929332733,
  0.0663476511836052,
  0.05931428447365761,
  0.05503835529088974,
  0.052045680582523346],
 'sparse_categorical_accuracy': [0.6918749809265137,
  0.8012499809265137,
  0.8462499976158142,
  0.8887500166893005,
  0.909375011920929,
  0.909375011920929,
  0.9243749976158142,
  0.9293749928474426,
  0.9381250143051147,
  0.9556249976158142,
  0.9568750262260437,
  0.9637500047683716,
  0.9731249809265137,
  0.9700000286102295,
  0.9756249785423279,
  0.9762499928474426,
  0.981249988079071,
  0.9818750023841858,
  0.984375,
  0.9825000166893005],
 'val_loss': [0.6298846006393433,
  0.39916881918907166,
  0.3609302043914795,
  1.

### Evaluate the Model

In [92]:
training_loss = history.history['loss'][-1]
training_accuracy = history.history['sparse_categorical_accuracy'][-1]
val_loss = history.history['val_loss'][-1]
val_accuracy = history.history['val_sparse_categorical_accuracy'][-1]

test_loss, test_accuracy = model.evaluate(test_ds)

print(f'Training Loss: {training_loss:.4f}')
print(f'Training Accuracy: {training_accuracy*100.:.2f}%')
print()
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy*100.:.2f}%')
print()
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy*100.:.2f}%')

10/10 [==============================] - 0s 9ms/step - loss: 0.0739 - sparse_categorical_accuracy: 0.9650
Training Loss: 0.0520
Training Accuracy: 98.25%

Validation Loss: 0.0655
Validation Accuracy: 97.50%

Test Loss: 0.0739
Test Accuracy: 96.50%


## ResNet8

In [93]:
class ConvReluBN(tf.keras.layers.Layer):
    def __init__(self, num_outputs, kernel_size, stride,bn=False):
        super().__init__()
        self.bn = bn
        self.stride = stride
        
        self.conv = tf.keras.layers.Conv2D(num_outputs, kernel_size, strides=stride, padding="same", use_bias=not bn)
        self.relu = tf.keras.layers.ReLU()
        if bn:
            self.bn_layer = tf.keras.layers.BatchNormalization()

    def call(self, inputs, training=False):
        net = self.conv(inputs)
        net = self.relu(net)
        if self.bn:
            net = self.bn_layer(net, training=training)
        return net


class ResBlock(tf.keras.layers.Layer):
    def __init__(self, num_channels):
        super().__init__()
        self.conv_relu_bn_1 = ConvReluBN(num_channels, kernel_size=3, stride=1, bn=True)
        self.conv_relu_bn_2 = ConvReluBN(num_channels, kernel_size=3, stride=1, bn=False)
        self.bn_layer = tf.keras.layers.BatchNormalization()

    def call(self, inputs, training=False):
        layer_in = inputs
        net = self.conv_relu_bn_1(inputs, training=training)
        net = self.conv_relu_bn_2(net, training=training)
        net += layer_in
        net = self.bn_layer(net, training=training)
        return net

class ResNet(tf.keras.Model):
    def __init__(self, num_classes, num_blocks, num_channels):
        super().__init__()

        self.first_conv = tf.keras.layers.Conv2D(num_channels, 3, strides=2, padding='same')

        self.blocks = [ResBlock(num_channels) for i in range(num_blocks)]

        self.global_pool = tf.keras.layers.GlobalAveragePooling2D()
        self.fc = tf.keras.layers.Dense(num_classes)
        self.softmax = tf.keras.layers.Softmax()

    def call(self, inputs, training=False):
        x = self.first_conv(inputs)

        for block in self.blocks:
            x = block(x, training=training)

        x = self.global_pool(x)
        x = self.fc(x)
        x = self.softmax(x)

        return x


class ResNet8(ResNet):
    def __init__(self, num_classes):
        super().__init__(num_classes, num_blocks=3, num_channels=64)

In [99]:
model = ResNet8(num_classes = len(LABELS))

for example_batch, example_labels in train_ds.take(1):
  train_shp = example_batch.shape

model.build(train_shp)

In [100]:
model.summary()

Model: "res_net8_10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_64 (Conv2D)          multiple                  640       
                                                                 
 res_block_18 (ResBlock)     multiple                  74304     
                                                                 
 res_block_19 (ResBlock)     multiple                  74304     
                                                                 
 res_block_20 (ResBlock)     multiple                  74304     
                                                                 
 global_average_pooling2d_1  multiple                  0         
 2 (GlobalAveragePooling2D)                                      
                                                                 
 dense_12 (Dense)            multiple                  130       
                                                       

In [101]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=False)
optimizer = tf.optimizers.Adam(learning_rate=linear_decay)
metrics = [tf.metrics.SparseCategoricalAccuracy()]
model.compile(loss=loss, optimizer=optimizer, metrics=metrics)

history = model.fit(
    train_ds, 
    epochs=TRAINING_ARGS['epochs'], 
    validation_data=val_ds, 
    callbacks=[lr_scheduler, early_stopping]
)

Epoch 1/20
80/80 [==============================] - 7s 25ms/step - loss: 0.5407 - sparse_categorical_accuracy: 0.7644 - val_loss: 0.9482 - val_sparse_categorical_accuracy: 0.6750 - lr: 0.0095
Epoch 2/20
80/80 [==============================] - 2s 19ms/step - loss: 0.3788 - sparse_categorical_accuracy: 0.8569 - val_loss: 1.6102 - val_sparse_categorical_accuracy: 0.7650 - lr: 0.0090
Epoch 3/20
80/80 [==============================] - 2s 20ms/step - loss: 0.2847 - sparse_categorical_accuracy: 0.8963 - val_loss: 0.3592 - val_sparse_categorical_accuracy: 0.8900 - lr: 0.0085
Epoch 4/20
80/80 [==============================] - 2s 21ms/step - loss: 0.1887 - sparse_categorical_accuracy: 0.9331 - val_loss: 0.1611 - val_sparse_categorical_accuracy: 0.9550 - lr: 0.0080
Epoch 5/20
80/80 [==============================] - 2s 20ms/step - loss: 0.1127 - sparse_categorical_accuracy: 0.9575 - val_loss: 0.1850 - val_sparse_categorical_accuracy: 0.9500 - lr: 0.0075
Epoch 6/20
80/80 [======================

In [102]:
history.history

{'loss': [0.5406778454780579,
  0.3788401484489441,
  0.28474974632263184,
  0.18873101472854614,
  0.11273398250341415,
  0.1116304025053978,
  0.08590099215507507,
  0.09046763181686401,
  0.07532865554094315,
  0.06167248636484146,
  0.04645991325378418,
  0.03291372209787369,
  0.02882673777639866,
  0.02635963261127472,
  0.020871076732873917,
  0.018159527331590652,
  0.01593026891350746,
  0.014607511460781097,
  0.013584290631115437,
  0.012155568227171898],
 'sparse_categorical_accuracy': [0.7643749713897705,
  0.8568750023841858,
  0.8962500095367432,
  0.9331250190734863,
  0.9574999809265137,
  0.9587500095367432,
  0.9674999713897705,
  0.9643750190734863,
  0.96875,
  0.9756249785423279,
  0.9800000190734863,
  0.9868749976158142,
  0.9887499809265137,
  0.9887499809265137,
  0.9918749928474426,
  0.9931250214576721,
  0.9931250214576721,
  0.9925000071525574,
  0.9925000071525574,
  0.9956250190734863],
 'val_loss': [0.9482169151306152,
  1.6102064847946167,
  0.35921937

In [103]:
training_loss = history.history['loss'][-1]
training_accuracy = history.history['sparse_categorical_accuracy'][-1]
val_loss = history.history['val_loss'][-1]
val_accuracy = history.history['val_sparse_categorical_accuracy'][-1]

test_loss, test_accuracy = model.evaluate(test_ds)

print(f'Training Loss: {training_loss:.4f}')
print(f'Training Accuracy: {training_accuracy*100.:.2f}%')
print()
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy*100.:.2f}%')
print()
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy*100.:.2f}%')

10/10 [==============================] - 0s 9ms/step - loss: 0.0622 - sparse_categorical_accuracy: 0.9700
Training Loss: 0.0122
Training Accuracy: 99.56%

Validation Loss: 0.0396
Validation Accuracy: 98.50%

Test Loss: 0.0622
Test Accuracy: 97.00%


### Save the model

In [ ]:
import os
from time import time

timestamp = int(time())

saved_model_dir = f'./saved_models/{timestamp}'
if not os.path.exists(saved_model_dir):
    os.makedirs(saved_model_dir)
model.save(saved_model_dir)



### Save Hyperparameters & Results

On the left sidebar, set "Incoming connections" to "On". Tensorboard will be available at the provided link.

In [ ]:
import pandas as pd

output_dict = {
    'timestamp': timestamp,
    **PREPROCESSING_ARGS,
    **TRAINING_ARGS,
    'test_accuracy': test_accuracy
}

df = pd.DataFrame([output_dict])

output_path='./mel_spectrogram_results.csv'
df.to_csv(output_path, mode='a', header=not os.path.exists(output_path), index=False)

### TFLite Conversion

In [ ]:
# Converting a SavedModel to a TensorFlow Lite model.
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
tflite_model = converter.convert()

tflite_model_dir = './tflite_models'
# create the path if it doesn't exist
if not os.path.exists(tflite_model_dir):
    os.makedirs(tflite_model_dir)

# name the model
tflite_model_name = os.path.join(tflite_model_dir, f'{timestamp}.tflite')
tflite_model_name

# write the model
with open(tflite_model_name, 'wb') as fp:
    fp.write(tflite_model)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b4ef5aa4-3f71-4837-91f1-c6fd9810a7ea' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>